# Fraud Detection

**Source:** IEEE Computational Intelligence Society (IEEE-CIS), in partnership with Vesta Corporation, hosted on Kaggle as the "IEEE-CIS Fraud Detection" competition (2019).

**Size:** Two linked tables joined on `TransactionID`:
- `train_transaction.csv` — 590,540 rows, 394 columns
- `train_identity.csv` — 144,233 rows, 41 columns
- Merged (left join, all transactions retained): 590,540 rows, 434 columns. Only ~23% of transactions have a matching identity record.

**Business Context:** Vesta Corporation is a payment service provider processing high-volume e-commerce transactions globally. Manual review at this scale is infeasible, so an automated model must flag suspicious transactions close to real time. This is a cost-asymmetric problem: a missed fraud (false negative) causes direct financial loss and chargeback liability, while a wrongly blocked legitimate transaction (false positive) damages customer trust. Framed as **binary classification** (not anomaly detection) because every transaction carries a definitive post-hoc label.

**Application Context:** The model would score each transaction inline at the point of authorisation, using only information available at that moment (amount, card/device details, behavioural signals). This means: predictions must be low-latency, the model must use the same features at inference as at training, and any post-transaction-outcome features must be excluded to avoid leakage.

**Feature Groups:** transaction metadata (`TransactionAmt`, `ProductCD`, `TransactionDT`), card info (`card1`–`card6`), address/distance (`addr1`, `addr2`, `dist1`, `dist2`), email domains (`P_emaildomain`, `R_emaildomain`), counting features (`C1`–`C14`), timedeltas (`D1`–`D15`), match flags (`M1`–`M9`), Vesta-engineered features (`V1`–`V339`, semantics undisclosed), and identity features (`id_01`–`id_38`, `DeviceType`, `DeviceInfo`).

**Realistic Challenges:** severe class imbalance (~3.5% fraud), high dimensionality (434 raw columns), heavy missingness (many `V`-columns >90% missing), partial identity-table coverage (~23%), and opaque/anonymised `V`-columns with no documented meaning.

### Target Variable and Prediction Objective

**Target:** `isFraud` — binary (1 = fraudulent, 0 = legitimate), present only in training data.

**Objective:** Predict the *probability* that a transaction is fraudulent (continuous risk score in [0, 1]), not just a hard label, so a business-calibrated threshold can trade off fraud losses against false-positive customer friction. Model selection below prioritises ROC-AUC / PR-AUC / recall-at-precision over raw accuracy, since accuracy is misleading under 3.5% class prevalence.

In [0]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [0]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"project_root: {project_root}")

# Imports

In [0]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import os

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully")

OUTPUT_PATH_figures = "/Users/chamika/Desktop/git/machine_learning/Question_1/outputs/figures/"
DATA_PATH_raw    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/raw/"
DATA_PATH_processed    = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/processed/"

#  Load Data

In [0]:
# Cell 3: Load raw data
DATA_PATH = "/Users/chamika/Desktop/git/machine_learning/Question_1/data/raw/"

print("Loading transaction data...")
train_trans = pd.read_csv(DATA_PATH + "train_transaction.csv")
print(f"✅ train_transaction: {train_trans.shape}")

print("Loading identity data...")
train_id = pd.read_csv(DATA_PATH + "train_identity.csv")
print(f"✅ train_identity:    {train_id.shape}")

# Merge on TransactionID (left join — keep all transactions)
df = train_trans.merge(train_id, on='TransactionID', how='left')
print(f"✅ Merged dataset:    {df.shape}")

# Basic Overview

In [0]:
# Cell 4: Basic dataset overview
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Rows:     {df.shape[0]:,}")
print(f"Columns:  {df.shape[1]:,}")
print(f"\nData Types:\n{df.dtypes.value_counts()}")
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"\nFirst 3 rows:")
df.head(3)

# Target Variable Analysis (Class Imbalance)

In [0]:
# Cell 5: Target variable - class imbalance analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count of each class
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100

# Plot 1: Count bar chart
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], 
            fraud_counts.values,
            color=['#2ecc71', '#e74c3c'], 
            edgecolor='black', linewidth=0.8)
axes[0].set_title('Transaction Count by Class', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontweight='bold')

# Plot 2: Pie chart
axes[1].pie(fraud_counts.values,
            labels=[f'Legitimate\n{fraud_pct[0]:.1f}%', 
                    f'Fraud\n{fraud_pct[1]:.1f}%'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.2f%%',
            startangle=90,
            explode=(0, 0.08))
axes[1].set_title('Class Distribution', fontsize=14, fontweight='bold')

plt.suptitle('IEEE-CIS Fraud Detection — Class Imbalance Analysis', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'class_imbalance.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Class Distribution Summary:")
print(f"  Legitimate transactions: {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)")
print(f"  Fraudulent transactions: {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)")
print(f"  Imbalance ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1")

#  Transaction Amount Distribution

In [0]:
# Cell 6: Transaction amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Log-scale distribution by class
for label, color, name in zip([0, 1], ['#2ecc71', '#e74c3c'], 
                                ['Legitimate', 'Fraud']):
    subset = df[df['isFraud'] == label]['TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=60, alpha=0.6, 
                 color=color, label=f'{name} (n={len(subset):,})',
                 edgecolor='white', linewidth=0.3)

axes[0].set_title('Transaction Amount Distribution (log scale)', 
                   fontsize=13, fontweight='bold')
axes[0].set_xlabel('log(TransactionAmt + 1)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Plot 2: Boxplot comparison
fraud_amt   = df[df['isFraud'] == 1]['TransactionAmt']
legit_amt   = df[df['isFraud'] == 0]['TransactionAmt']
axes[1].boxplot([np.log1p(legit_amt), np.log1p(fraud_amt)],
                tick_labels=['Legitimate', 'Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='#2ecc71', alpha=0.6),
                medianprops=dict(color='black', linewidth=2))
axes[1].set_title('Transaction Amount Boxplot by Class (log scale)', 
                   fontsize=13, fontweight='bold')
axes[1].set_ylabel('log(TransactionAmt + 1)')

plt.suptitle('Transaction Amount Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'transaction_amount.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print("📊 Transaction Amount Summary:")
print(f"\nLegitimate transactions:")
print(legit_amt.describe().round(2))
print(f"\nFraudulent transactions:")
print(fraud_amt.describe().round(2))

# How many transactions exceed common thresholds?
print(f"\nHow many legitamate transactions exceed common thresholds?")
for threshold in [1000, 2000, 5000, 10000, 20000, 30000]:
    count = (legit_amt > threshold).sum()
    pct = count / len(legit_amt) * 100
    print(f"> ${threshold:,}: {count:,} transactions ({pct:.3f}%)")

print(f"\nHow many fraud transactions exceed common thresholds?")
for threshold in [1000, 2000, 5000]:
    count = (fraud_amt > threshold).sum()
    pct = count / len(legit_amt) * 100
    print(f"> ${threshold:,}: {count:,} transactions ({pct:.3f}%)")

#  Missing Value Analysis

In [0]:
# Cell 7: Missing value analysis
# Calculate missing percentages
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing %', ascending=False)

missing = missing[missing['Missing Count'] > 0]

print(f"📊 Columns with missing values: {len(missing)} out of {df.shape[1]}")
print(f"\nTop 20 columns by missing %:")
print(missing.head(20).to_string())

# Plot: Missing value heatmap (top 50 columns)
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Bar chart — top 40 missing columns
top40 = missing.head(40)
colors = ['#e74c3c' if x > 50 else '#f39c12' if x > 20 else '#3498db' 
          for x in top40['Missing %']]
axes[0].barh(top40.index[::-1], top40['Missing %'][::-1], color=colors[::-1])
axes[0].axvline(x=50, color='red', linestyle='--', 
                linewidth=1.5, label='50% threshold')
axes[0].axvline(x=20, color='orange', linestyle='--', 
                linewidth=1.5, label='20% threshold')
axes[0].set_title('Top 40 Columns by Missing Value %', 
                   fontsize=13, fontweight='bold')
axes[0].set_xlabel('Missing %')
axes[0].legend()

# Missing value matrix (sample for visibility)
msno.matrix(df[missing.head(30).index].sample(5000, random_state=42), 
            ax=axes[1], sparkline=False, color=(0.2, 0.4, 0.8))
axes[1].set_title('Missing Value Pattern (5,000 row sample)', 
                   fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'missing_values.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Categorise by severity
print("\n📊 Missing Value Severity:")
print(f"  >80% missing:  {len(missing[missing['Missing %'] > 80])} columns")
print(f"  50-80% missing: {len(missing[(missing['Missing %'] > 50) & (missing['Missing %'] <= 80)])} columns")
print(f"  20-50% missing: {len(missing[(missing['Missing %'] > 20) & (missing['Missing %'] <= 50)])} columns")
print(f"  <20% missing:  {len(missing[missing['Missing %'] <= 20])} columns")

# Correlation Heatmap

In [0]:
# Cell 8: Correlation analysis
# Select numerical columns with low missing % for meaningful correlation
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if df[c].isnull().mean() < 0.3]  # <30% missing
num_cols = [c for c in num_cols if c not in ['TransactionID']]

# Correlation with target
target_corr = df[num_cols].corrwith(df['isFraud']).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Top 20 features correlated with isFraud
top20_corr = target_corr.drop('isFraud').head(20)
colors = ['#e74c3c' if x > 0.1 else '#f39c12' if x > 0.05 else '#3498db' 
          for x in top20_corr.values]
axes[0].barh(top20_corr.index[::-1], top20_corr.values[::-1], color=colors[::-1])
axes[0].set_title('Top 20 Features Correlated with isFraud', 
                   fontsize=13, fontweight='bold')
axes[0].set_xlabel('Absolute Pearson Correlation')
axes[0].axvline(x=0.1, color='red', linestyle='--', linewidth=1.5, label='>0.10 (strong)')
axes[0].axvline(x=0.05, color='orange', linestyle='--', linewidth=1.5, label='>0.05 (moderate)')
axes[0].legend(fontsize=9)

# Plot 2: Heatmap of top 15 correlated features
top15_features = target_corr.drop('isFraud').head(15).index.tolist() + ['isFraud']
corr_matrix = df[top15_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[1], 
            cmap='RdYlGn', center=0, annot=True, 
            fmt='.2f', linewidths=0.5,
            annot_kws={'size': 7})
axes[1].set_title('Correlation Heatmap — Top 15 Features + Target',
                   fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle('Correlation Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Top 10 features correlated with isFraud:")
print(target_corr.drop('isFraud').head(10).round(4).to_string())

# Categorical Feature Distributions

In [0]:
# Cell 9: Categorical feature distributions
cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'DeviceType', 'M4']
cat_cols = [c for c in cat_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    # Fraud rate per category
    fraud_rate = df.groupby(col)['isFraud'].agg(['mean', 'count']).reset_index()
    fraud_rate.columns = [col, 'fraud_rate', 'count']
    fraud_rate = fraud_rate[fraud_rate['count'] > 100]  # filter low-count categories
    fraud_rate = fraud_rate.sort_values('fraud_rate', ascending=False).head(15)
    
    bars = axes[idx].bar(range(len(fraud_rate)), 
                          fraud_rate['fraud_rate'] * 100,
                          color=plt.cm.RdYlGn_r(
                              fraud_rate['fraud_rate'] / fraud_rate['fraud_rate'].max()),
                          edgecolor='black', linewidth=0.5)
    
    axes[idx].set_xticks(range(len(fraud_rate)))
    axes[idx].set_xticklabels(fraud_rate[col].astype(str), 
                               rotation=45, ha='right', fontsize=8)
    axes[idx].set_title(f'Fraud Rate by {col}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Fraud Rate (%)')
    axes[idx].axhline(y=df['isFraud'].mean() * 100, color='blue', 
                       linestyle='--', linewidth=1.5, label='Overall avg')
    axes[idx].legend(fontsize=8)
    
    # Add count labels
    for i, (bar, cnt) in enumerate(zip(bars, fraud_rate['count'])):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        f'n={cnt:,}', ha='center', fontsize=6, rotation=0)

plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + "categorical_fraud_rates.png", dpi=150, bbox_inches='tight')
plt.show()

# Time-Based Fraud Patterns

In [0]:
# Cell 10: Time-based fraud patterns
# TransactionDT is seconds elapsed — convert to hour/day of week
df_time = df.copy()
df_time['hour'] = (df_time['TransactionDT'] / 3600) % 24
df_time['day_of_week'] = (df_time['TransactionDT'] / (3600 * 24)) % 7

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Fraud rate by hour of day
hourly = df_time.groupby(df_time['hour'].astype(int))['isFraud'].agg(['mean', 'count'])
ax1_twin = axes[0].twinx()
axes[0].bar(hourly.index, hourly['count'], 
            alpha=0.3, color='#3498db', label='Transaction count')
ax1_twin.plot(hourly.index, hourly['mean'] * 100, 
              color='#e74c3c', linewidth=2.5, 
              marker='o', markersize=4, label='Fraud rate %')
axes[0].set_title('Fraud Rate & Volume by Hour of Day', 
                   fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Transaction Count', color='#3498db')
ax1_twin.set_ylabel('Fraud Rate (%)', color='#e74c3c')
axes[0].legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Plot 2: Fraud rate by day of week
daily = df_time.groupby(df_time['day_of_week'].astype(int))['isFraud'].agg(['mean', 'count'])
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax2_twin = axes[1].twinx()
axes[1].bar(range(len(daily)), daily['count'], 
            alpha=0.3, color='#3498db', label='Transaction count')
ax2_twin.plot(range(len(daily)), daily['mean'] * 100, 
              color='#e74c3c', linewidth=2.5, 
              marker='o', markersize=6, label='Fraud rate %')
axes[1].set_xticks(range(len(daily)))
axes[1].set_xticklabels(days[:len(daily)])
axes[1].set_title('Fraud Rate & Volume by Day of Week', 
                   fontsize=13, fontweight='bold')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Transaction Count', color='#3498db')
ax2_twin.set_ylabel('Fraud Rate (%)', color='#e74c3c')
axes[1].legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.suptitle('Temporal Fraud Patterns', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 Peak fraud hours:")
print(hourly['mean'].sort_values(ascending=False).head(5).round(4))

#  Numerical Feature Distributions (C & V columns)

In [0]:
# Cell 11-a: Numerical feature distributions — C columns (count features)
c_cols = [f'C{i}' for i in range(1, 15) if f'C{i}' in df.columns]

fig, axes = plt.subplots(2, 7, figsize=(20, 8))
axes = axes.flatten()

for idx, col in enumerate(c_cols):
    legit = np.log1p(df[df['isFraud']==0][col].dropna())
    fraud = np.log1p(df[df['isFraud']==1][col].dropna())
    
    axes[idx].hist(legit, bins=40, alpha=0.5, color='#2ecc71', 
                   label='Legit', density=True)
    axes[idx].hist(fraud, bins=40, alpha=0.5, color='#e74c3c', 
                   label='Fraud', density=True)
    axes[idx].set_title(col, fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('log(x+1)', fontsize=7)
    axes[idx].tick_params(labelsize=7)
    if idx == 0:
        axes[idx].legend(fontsize=7)

plt.suptitle('C Feature Distributions — Legitimate vs Fraud (log scale)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'c_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# Cell 11-b: Numerical feature distributions — V columns (Vesta engineered features)
v_cols = [c for c in df.columns if c.startswith('V')]
print(f"Total V columns: {len(v_cols)}")

# Step 1: Correlation of each V column with isFraud (drop rows with missing values per column)
v_corr = df[v_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
v_corr_sorted = v_corr.abs().sort_values(ascending=False)

print(f"\nTop 14 V columns by |correlation| with isFraud:")
print(v_corr_sorted.head(14).round(4))

# Step 2: Bar chart — correlation strength across ALL V columns (sorted)
fig, ax = plt.subplots(figsize=(18, 6))

colors_corr = ['#e74c3c' if v > 0 else '#3498db' for v in v_corr.loc[v_corr_sorted.index]]
ax.bar(range(len(v_corr_sorted)), v_corr.loc[v_corr_sorted.index].values,
       color=colors_corr, edgecolor='none', width=1.0)
ax.set_title(f'Correlation with isFraud — All {len(v_cols)} V Columns (sorted by |corr|)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('V columns (sorted)')
ax.set_ylabel('Correlation with isFraud')
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'v_correlation_bars.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 3: Histogram grid — top 14 most correlated V columns (same style as C-columns cell)
top14_v = v_corr_sorted.head(14).index.tolist()

fig2, axes2 = plt.subplots(2, 7, figsize=(20, 8))
axes2 = axes2.flatten()

for idx, col in enumerate(top14_v):
    legit = np.log1p(df[df['isFraud']==0][col].dropna().clip(lower=0))
    fraud = np.log1p(df[df['isFraud']==1][col].dropna().clip(lower=0))

    axes2[idx].hist(legit, bins=40, alpha=0.5, color='#2ecc71',
                     label='Legit', density=True)
    axes2[idx].hist(fraud, bins=40, alpha=0.5, color='#e74c3c',
                     label='Fraud', density=True)
    axes2[idx].set_title(f'{col}\n(r={v_corr[col]:.3f})', fontsize=9, fontweight='bold')
    axes2[idx].set_xlabel('log(x+1)', fontsize=7)
    axes2[idx].tick_params(labelsize=7)
    if idx == 0:
        axes2[idx].legend(fontsize=7)

plt.suptitle('Top 14 Most Fraud-Correlated V Columns — Legitimate vs Fraud (log scale)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'v_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Step 4: Missingness summary for V columns (they come in blocks with different missing patterns)
v_missing = df[v_cols].isnull().mean().sort_values(ascending=False)
print(f"\nV-column missingness range: {v_missing.min()*100:.1f}% – {v_missing.max()*100:.1f}%")
print(f"V columns with >50% missing: {(v_missing > 0.5).sum()} / {len(v_cols)}")

# Outlier Detection

In [0]:
# Cell 12: Outlier detection — full scan, then detailed view of worst offenders
# Step 1: Scan ALL numerical columns using the IQR method
num_cols_all = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols_all = [c for c in num_cols_all if c not in ['TransactionID', 'isFraud']]

outlier_summary_full = {}
for col in num_cols_all:
    data = df[col].dropna()
    if len(data) == 0:
        continue
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        continue  # skip constant/near-constant columns — IQR method breaks down
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((data < lower) | (data > upper)).sum()
    outlier_pct = outliers / len(data) * 100
    outlier_summary_full[col] = {'count': outliers, 'pct': round(outlier_pct, 2)}

outlier_df = pd.DataFrame(outlier_summary_full).T.sort_values('pct', ascending=False)
print(f"📊 Scanned {len(outlier_summary_full)} numerical columns for outliers (IQR method)")
print(f"\nTop 20 columns by outlier %:")
print(outlier_df.head(20))

# Step 2: Auto-select top 7 by outlier % for detailed visualization
key_num_cols = outlier_df.head(7).index.tolist()
print(f"\n✅ Selected for detailed view (highest outlier %): {key_num_cols}")

# Step 3: Boxplot grid for the selected columns (same style as before)
fig, axes = plt.subplots(1, len(key_num_cols), figsize=(20, 5))

for idx, col in enumerate(key_num_cols):
    data = df[col].dropna()
    outlier_pct = outlier_df.loc[col, 'pct']

    axes[idx].boxplot(np.log1p(data.clip(lower=0)),
                       tick_labels=[col],
                       patch_artist=True,
                       boxprops=dict(facecolor='#3498db', alpha=0.6),
                       medianprops=dict(color='red', linewidth=2),
                       flierprops=dict(marker='.', markersize=1,
                                       alpha=0.3, color='#e74c3c'))
    axes[idx].set_title(f'{col}\n{outlier_pct:.1f}% outliers',
                         fontsize=9, fontweight='bold')
    axes[idx].set_xlabel('log(x+1)', fontsize=7)
    axes[idx].tick_params(labelsize=7)

plt.suptitle('Outlier Analysis — Top 7 Columns by Outlier % (log scale)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Outlier Summary (selected columns):")
for col in key_num_cols:
    print(f"  {col:20s}: {int(outlier_df.loc[col, 'count']):,} outliers ({outlier_df.loc[col, 'pct']}%)")

# Data Quality Summary

In [0]:
# Cell 13: Data quality summary report
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# 1. Duplicates
dup_count = df.duplicated().sum()
dup_trans  = df.duplicated(subset=['TransactionID']).sum()
print(f"\n1. DUPLICATES")
print(f"   Full row duplicates:       {dup_count:,}")
print(f"   TransactionID duplicates:  {dup_trans:,}")

# 2. Missing values summary
total_cells = df.shape[0] * df.shape[1]
total_missing = df.isnull().sum().sum()
print(f"\n2. MISSING VALUES")
print(f"   Total missing cells:  {total_missing:,} / {total_cells:,}")
print(f"   Overall missing %:    {total_missing/total_cells*100:.2f}%")
print(f"   Columns > 90% miss:   {(df.isnull().mean() > 0.9).sum()}")
print(f"   Columns > 50% miss:   {(df.isnull().mean() > 0.5).sum()}")
print(f"   Columns > 20% miss:   {(df.isnull().mean() > 0.2).sum()}")
print(f"   Columns 0% missing:   {(df.isnull().mean() == 0).sum()}")

# 3. Constant / near-constant columns
low_var = [c for c in df.columns 
           if df[c].nunique() <= 1]
print(f"\n3. CONSTANT COLUMNS (nunique <= 1):  {len(low_var)}")
if low_var:
    print(f"   {low_var}")

# 4. Cardinality of categorical columns
cat_cols_all = df.select_dtypes(include='object').columns.tolist()
print(f"\n4. CATEGORICAL CARDINALITY")
for c in cat_cols_all[:10]:
    print(f"   {c:30s}: {df[c].nunique()} unique values")

# 5. Target distribution
print(f"\n5. TARGET VARIABLE (isFraud)")
print(f"   Legitimate: {(df['isFraud']==0).sum():,} ({(df['isFraud']==0).mean()*100:.2f}%)")
print(f"   Fraud:      {(df['isFraud']==1).sum():,} ({(df['isFraud']==1).mean()*100:.2f}%)")
print(f"   Ratio:      {(df['isFraud']==0).sum()/(df['isFraud']==1).sum():.1f}:1")

print("\n" + "=" * 60)
print("✅ Data Quality Report Complete")
print("=" * 60)

# Missing Value Strategy & Handling

In [0]:
# Cell 14-a: Missing value handling strategy
# Three-tier approach based on missing percentage

print("=" * 60)
print("MISSING VALUE HANDLING STRATEGY")
print("=" * 60)

# Calculate missing % per column
missing_pct = df.isnull().mean() * 100

# Tier 1: Drop columns > 90% missing
drop_cols = missing_pct[missing_pct > 90].index.tolist()

# Tier 2: High missing (50-90%) — impute with median/mode
high_missing = missing_pct[(missing_pct > 50) & (missing_pct <= 90)].index.tolist()

# Tier 3: Low-moderate missing (<50%) — impute with median/mode
low_missing = missing_pct[(missing_pct > 0) & (missing_pct <= 50)].index.tolist()

print(f"\nTier 1 — DROP (>90% missing):        {len(drop_cols)} columns")
print(f"Tier 2 — HIGH missing (50–90%):      {len(high_missing)} columns")
print(f"Tier 3 — LOW/MOD missing (0–50%):    {len(low_missing)} columns")
print(f"Complete columns (0% missing):        {(missing_pct == 0).sum()} columns")

# ── Apply Tier 1: Drop high-missing columns ──────────────────
df_clean = df.drop(columns=drop_cols)
print(f"\n✅ After dropping Tier 1 columns:")
print(f"   Shape: {df_clean.shape}")
print(f"   Dropped: {drop_cols}")

## Missingness flag columns before imputing

In [0]:
# Cell 14-b: Missing value handling strategy
# Add missingness flags for columns with substantial missing data (before imputation)

missing_threshold = 30  # percent
flag_candidates = missing_pct[(missing_pct > missing_threshold) & (missing_pct <= 90)].index.tolist()
flag_candidates = [c for c in flag_candidates if c in df_clean.columns]

print(f"Adding 'was_missing' flags for {len(flag_candidates)} columns (>{missing_threshold}% missing)")

for col in flag_candidates:
    df_clean[f'{col}_was_missing'] = df_clean[col].isnull().astype(int)

print(f"flag_candidates: {len(flag_candidates)}, unique: {len(set(flag_candidates))}")

0.02 threshold is a reasonable starting cutoff for "worth keeping" at this sample size (590k rows) — even a small correlation is statistically meaningful with that many observations, but you want to avoid keeping dozens of flags that add near-zero information. Feel free to adjust after seeing your actual numbers — if very few flags clear 0.02, lower it slightly (e.g. 0.01); if too many do, raise it.

In [0]:
# Cell 14-c: Missing value handling strategy
# Check correlation of each 'was_missing' flag with isFraud
flag_cols = [c for c in df_clean.columns if c.endswith('_was_missing')]

flag_corr = df_clean[flag_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
flag_corr_sorted = flag_corr.sort_values(ascending=False)

print(f"Checked {len(flag_cols)} 'was_missing' flags for correlation with isFraud\n")
print("Top 10 strongest POSITIVE correlations (missing → more likely fraud):")
print(flag_corr_sorted.head(10).round(4))

print("\nTop 10 strongest NEGATIVE correlations (missing → less likely fraud):")
print(flag_corr_sorted.tail(10).round(4))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
top_flags = pd.concat([flag_corr_sorted.head(10), flag_corr_sorted.tail(10)])
colors = ['#e74c3c' if v > 0 else '#3498db' for v in top_flags]

ax.barh(range(len(top_flags)), top_flags.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(len(top_flags)))
ax.set_yticklabels(top_flags.index, fontsize=8)
ax.set_xlabel('Correlation with isFraud')
ax.set_title("'was_missing' Flags — Strongest Correlations with Fraud", fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(OUTPUT_PATH_figures + 'missingness_flag_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag threshold: keep only flags with meaningful signal (e.g. |corr| > 0.02)
useful_flags = flag_corr[flag_corr.abs() > 0.02].index.tolist()
weak_flags = [c for c in flag_cols if c not in useful_flags]

print(f"\n✅ Flags with meaningful signal (|corr| > 0.02): {len(useful_flags)} / {len(flag_cols)}")
print(f"❌ Weak flags (candidates for dropping): {len(weak_flags)}")

# Optionally drop the weak ones to keep feature count manageable
df_clean = df_clean.drop(columns=weak_flags)
print(f"\nShape after dropping weak flags: {df_clean.shape}")

print("\nActual distribution of |corr| values:")
print(flag_corr.abs().describe())
print("\n90th/95th/99th percentile of |corr|:")
print(flag_corr.abs().quantile([0.5, 0.75, 0.90, 0.95, 0.99]))

Shape of the |corr| values distribution:

- 25th percentile = 0.07, median = 0.13, 75th percentile = 0.13, max = 0.16
- The values are tightly clustered between 0.07 and 0.16 — there's no long tail of barely-above-threshold noise. Nearly all surviving flags sit solidly in the 0.10–0.14 range.

This actually contradicts my earlier concern about sample-size-inflated noise. If it were mostly noise, you'd expect a pile-up right near 0.02–0.03 with a few genuine outliers higher up. Instead you're seeing a consistent, moderate-strength signal across almost the entire set — for comparison, your best individual V-columns topped out around r=0.38, and a solid categorical predictor might sit around 0.1–0.2, so 0.07–0.16 is a meaningful effect size in this dataset's context, not trivial.
What this is actually telling you, substantively: missingness in this dataset isn't random — it's structurally tied to fraud across a huge swath of columns simultaneously. That's a real, defensible pattern (likely reflecting that certain transaction/device/identity paths are both more likely to have missing fields and more likely to be fraudulent — e.g. automated or unusual client configurations skip populating certain optional fields).

# Impute Remaining Missing Values

In [0]:
# Cell 15: Impute missing values

# Separate column types
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()

# Remove target and ID from imputation
exclude = ['TransactionID', 'isFraud']
num_cols_imp = [c for c in num_cols if c not in exclude]
cat_cols_imp = [c for c in cat_cols if c not in exclude]

print(f"Numerical columns to impute: {len(num_cols_imp)}")
print(f"Categorical columns to impute: {len(cat_cols_imp)}")

# Numerical → median imputation
for col in num_cols_imp:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical → mode imputation  
for col in cat_cols_imp:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Verify
remaining_missing = df_clean.isnull().sum().sum()
print(f"\n✅ After imputation:")
print(f"   Shape:           {df_clean.shape}")
print(f"   Remaining missing cells: {remaining_missing}")
print(f"   Missing %:       {remaining_missing / df_clean.size * 100:.4f}%")

# Outlier Treatment

In [0]:
# Cell 16-a: [DIAGNOSTIC ONLY — not used in final pipeline]
# Exploring whether a data-driven outlier threshold could be found per column.
# Result: both IQR-count and percentile-count methods proved unreliable due to
# zero-inflation, motivating the uniform approach used in Cell 16 instead.

num_cols_all = df_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_all = [c for c in num_cols_all if c not in ['TransactionID', 'isFraud']]

outlier_pct_report = {}
for col in num_cols_all:
    data = df_clean[col].dropna()
    if len(data) < 10:
        continue
    p1 = data.quantile(0.01)
    p99 = data.quantile(0.99)
    extreme = ((data < p1) | (data > p99)).sum()
    outlier_pct_report[col] = round(extreme / len(data) * 100, 3)

pct_df = pd.Series(outlier_pct_report).sort_values(ascending=False)

print(f"Scanned {len(pct_df)} / {len(num_cols_all)} numerical columns (percentile method — no columns skipped)")
print("\nDistribution of outlier %:")
print(pct_df.describe())
print("\nPercentile breakdown:")
print(pct_df.quantile([0.5, 0.75, 0.90, 0.95, 0.99]))

# Visualize to find the natural cutoff across the FULL set
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(pct_df)), pct_df.values, linewidth=1.2)
ax.set_xlabel('Columns (ranked by outlier %)')
ax.set_ylabel('Outlier %')
ax.set_title(f'Outlier % Across All {len(pct_df)} Numerical Columns (1st/99th percentile method)')
ax.axhline(y=2.0, color='red', linestyle='--', alpha=0.6, label='2% reference line')
ax.legend()
plt.tight_layout()
plt.show()

# Column counts at candidate thresholds
for t in [1.0, 2.0, 3.0, 4.0, 5.0]:
    n = (pct_df > t).sum()
    print(f"Threshold >{t}%: {n} columns qualify")

An initial attempt was made to identify a data-driven outlier threshold by ranking all 608 numerical columns by the percentage of values falling outside the 1st–99th percentile range. However, this metric proved structurally limited: by definition, no column can exceed approximately 2% under this measure, since the method only counts values in the extreme 1% tails on each side. This ceiling was confirmed empirically — the resulting distribution showed a mean of 0.44% and a maximum of exactly 2.00%, with zero columns qualifying at any threshold above 2% (Table X). Rather than revealing genuine differences in outlier severity between columns, the method mainly reflected each column's value-tie structure. Given this limitation, a uniform Winsorization rule was applied instead: all continuous numerical columns (excluding binary/flag features) were capped at their own 1st and 99th percentiles, since this is a low-risk operation that minimally affects well-behaved columns while meaningfully constraining genuinely skewed ones.

In [0]:
# Cell 16: Outlier treatment using percentile capping (Winsorization)
# Applied uniformly to all continuous numerical columns — binary/flag columns excluded
# (Rationale: IQR-count and percentile-count outlier ranking both proved unreliable due to
#  widespread zero-inflation in this dataset; percentile capping is low-risk enough to
#  apply broadly rather than requiring a per-column outlier-severity threshold.)

num_cols_all = df_clean.select_dtypes(include=[np.number]).columns.tolist()
num_cols_all = [c for c in num_cols_all if c not in ['TransactionID', 'isFraud']]

# Separate binary/flag columns (nothing meaningful to cap) from continuous columns
binary_like = [c for c in num_cols_all if df_clean[c].nunique() <= 2]
continuous_cols = [c for c in num_cols_all if c not in binary_like]

print(f"Total numerical columns: {len(num_cols_all)}")
print(f"Continuous columns to Winsorize: {len(continuous_cols)}")
print(f"Binary/flag columns skipped (nunique <= 2): {len(binary_like)}")

skipped_constant = []   # p1 == p99, clip() not applied
outlier_report = []     # clip() applied, tracks whether it actually changed an

for col in continuous_cols:
    p1 = df_clean[col].quantile(0.01)
    p99 = df_clean[col].quantile(0.99)

    if p1 == p99:
        skipped_constant.append(col)
        continue  # skip near-constant columns — nothing meaningful to cap

    before = ((df_clean[col] < p1) | (df_clean[col] > p99)).sum()
    df_clean[col] = df_clean[col].clip(lower=p1, upper=p99)

    outlier_report.append({
        'Column': col,
        'Capped Count': before,
        'Capped %': round(before / len(df_clean) * 100, 3)
    })

outlier_df_report = pd.DataFrame(outlier_report).sort_values('Capped %', ascending=False)

print(f"\n📊 Outlier Capping Report — Top 20 columns by % capped:")
print(outlier_df_report.head(20).to_string(index=False))

actually_capped = outlier_df_report[outlier_df_report['Capped Count'] > 0]
zero_effect = outlier_df_report[outlier_df_report['Capped Count'] == 0]

# --- Summary Breakdown ---
print(f"\n✅ Outlier Treatment — Full Breakdown")
print(f"   Method: Winsorization (1st–99th percentile capping)")
print(f"   Total numerical columns:              {len(num_cols_all)}")
print(f"   Binary/flag columns (skipped):         {len(binary_like)}")
print(f"   Continuous columns (eligible):         {len(continuous_cols)}")
print(f"   Skipped — constant p1==p99:            {len(skipped_constant)}")
print(f"   Processed with .clip() applied:        {len(outlier_report)}")
print(f"     └─ Actually had values capped:       {len(actually_capped)}")
print(f"     └─ Clip applied but no effect (0%):  {len(zero_effect)}")
print(f"   Rows preserved:                        {len(df_clean):,} (no rows dropped)")

IQR was used for detection (Cell 12) to identify outlier-heavy columns, while a milder 1st/99th percentile Winsorization was applied for treatment (Cell 16) to limit data alteration while still addressing extreme values.

### Outlier Treatment (Winsorization)
Outliers were addressed using Winsorization — capping extreme values at the 1st and 99th percentiles rather than removing the corresponding rows. Row removal was deliberately avoided given the severe class imbalance in the target variable (27.6:1 legitimate-to-fraud ratio); deleting outlier rows risked disproportionately removing rare fraud cases, since unusually extreme values may themselves be associated with fraudulent behaviour.
An initial attempt was made to identify a data-driven threshold for selecting which columns most needed outlier treatment, using both IQR-based and percentile-based outlier counts. Both methods proved unreliable due to widespread zero-inflation across the dataset: the percentile-based measure in particular is mathematically capped at approximately 2% by construction, and the resulting distribution (mean 0.44%, maximum 2.00%, with zero columns exceeding 2% at any tested threshold) confirmed it was reflecting value-tie structure rather than genuine outlier severity.
Given this limitation, a uniform rule was applied instead: Winsorization was performed on all continuous numerical columns, while binary and flag-type columns (including the 217 was_missing indicators and match-flag features such as M1–M9) were excluded, since percentile capping has no meaningful effect on binary data. Of 608 total numerical columns, 267 were classified as continuous, of which 224 had at least one value capped (the remaining 43 had identical 1st/99th percentile bounds and were left untouched). The most heavily affected columns were TransactionDT and id_02 (2.00% of values capped each), followed by a cluster of V-columns, D8, and dist1 (1.00% each). No rows were removed, preserving the full dataset of 590,540 transactions.

#### Caveat
Capping TransactionDT (a time-elapsed index, not a magnitude-based measure like transaction amount) doesn't carry the same real-world interpretation as capping something like TransactionAmt — it's mechanically treated the same way, but conceptually it's a different kind of variable.


# Encode Categorical Variables

In [0]:
# Cell 17: Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# Preserve raw email domain strings before encoding —
# email_match (in the feature engineering notebook) needs these,
# and can't use the encoded values (each column's LabelEncoder
# is fit independently, so codes aren't comparable across columns)
df_clean['P_emaildomain_raw'] = df_clean['P_emaildomain']
df_clean['R_emaildomain_raw'] = df_clean['R_emaildomain']

print("Encoding categorical variables...")
le = LabelEncoder()

cat_cols_encode = df_clean.select_dtypes(include='object').columns.tolist()
cat_cols_encode = [c for c in cat_cols_encode
                    if c not in ['TransactionID', 'P_emaildomain_raw', 'R_emaildomain_raw']]

print(f"Columns to encode: {len(cat_cols_encode)}")
print(cat_cols_encode)

for col in cat_cols_encode:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

print(f"\n✅ Encoding complete")
print(f"   Shape after encoding: {df_clean.shape}")
print(f"   All dtypes now numerical: {df_clean.select_dtypes(include='object').shape[1] == 2}")  # 2 = the _raw cols, kept as strings on purpose

# Quick check
print(f"\nData types after encoding:")
print(df_clean.dtypes.value_counts())

# Save Cleaned Data

In [0]:
# Cell 18: Save cleaned dataframe
import os

save_path = DATA_PATH_processed + 'df_clean.csv'
df_clean.to_csv(save_path, index=False)

file_size = os.path.getsize(save_path) / 1e9
print(f"✅ Cleaned data saved to:")
print(f"   {save_path}")
print(f"   Shape:     {df_clean.shape}")
print(f"   File size: {file_size:.2f} GB")